# Apply deterministic rules to PRESTAZIONI dataset

In [ ]:
import pandas as pd

### Define the correction function

In [ ]:
def apply_corrections(text, substitution_dict, words_to_delete):
    words = text.lower().split()
    corrected_words = []
    for word in words:
        if word in words_to_delete:
            continue
        corrected_word = substitution_dict.get(word, word)
        corrected_words.append(corrected_word)
    return " ".join(corrected_words)


### Load datasets

In [ ]:
# Load the nursing text dataset (tab-separated)
df = pd.read_csv("dataset/df_prestazioni_inf_clean_update.csv", sep="\t")

# Load the correction rules (semicolon-separated)
rules_df = pd.read_csv("vocabulary/vocab_FNOPI_corretto_istat.csv", sep=";")

### Prepare substitution dictionary and deletion list

In [ ]:
# Normalize relevant columns to lowercase
rules_df["word"] = rules_df["word"].str.lower()
rules_df["word_corretta"] = rules_df["word_corretta"].str.lower()

# Create substitution dictionary (exclude rows where 'word corretta' is 'eliminare')
substitutions_df = rules_df[rules_df["word_corretta"] != "eliminare"]
substitution_dict = dict(zip(substitutions_df["word"], substitutions_df["word_corretta"]))

# List of words to delete
words_to_delete = rules_df[rules_df["word_corretta"] == "eliminare"]["word"].tolist()


### Apply corrections to the dataset

In [ ]:
# Apply the correction function to the 'testo_pulito' column
df["testo_corretto"] = df["testo_pulito"].apply(
    lambda x: apply_corrections(x, substitution_dict, words_to_delete)
)


# Preview the result
df[["testo_pulito", "testo_corretto"]].head(50)


In [ ]:
df = df[['index', 'macroprestazione', 'testo', 'testo_pulito', 'testo_corretto', 'cod_valoremedio_liquidazione', 'descr_cod_valoremedio_liquidazione']]

# Save the cleaned dataset (optional)
df.to_csv("dataset/df_prestazioni_inf_corretto.csv", sep="\t", index=False)